# GRPO相关变体

本章节介绍GRPO算法的相关变体，包括DAPO、Dr.GRPO、BNPO、GSPO、GMPO、SAPO、LUSPO、SEED-GRPO、EDGE-GRPO、GAPO等。针对这些变体方法我们将其分为以下四类。

1. reward 如何变成 advantage；
2. token loss 如何做聚合与归一化；
3. importance ratio 在什么粒度上计算，以及 surrogate 如何约束；
4. 哪些样本、哪些 token 真正进入梯度。

围绕这四层，当前比较有代表性的工作可以分成下面四组。

## 分组一：归一化、优势形状与长度偏置修正

这一组讨论的是 reward 进入 loss 之前，应该怎样被缩放、重参数化，或者与回答长度解耦。它们针对的核心现象通常是长度偏置、长回答梯度稀释、以及 reward 归一化方式对更新强度的影响。

1. DAPO (2025-03) [DAPO: An Open-Source LLM Reinforcement Learning System at Scale](https://arxiv.org/abs/2503.14476)
2. Dr.GRPO (2025-03) [Understanding R1-Zero-Like Training: A Critical Perspective](https://arxiv.org/abs/2503.20783)
3. BNPO (2025-06) [BNPO: Beta Normalization Policy Optimization](https://arxiv.org/abs/2506.02864)

## 分组二：importance ratio 与 surrogate 形状改造

这一组默认 reward 已经给定，关注点转移到 policy shift 应该如何刻画。这里的主要问题是：ratio 应该按 token 计算还是按 sequence 计算；trust region 应该用硬裁剪、软门控，还是直接改 importance weight。

1. CISPO (2025-06) [MiniMax-M1: Scaling Test-Time Compute Efficiently with Lightning Attention](https://arxiv.org/abs/2506.13585)
2. GSPO (2025-07) [Group Sequence Policy Optimization](https://arxiv.org/abs/2507.18071)
3. GMPO (2025-07) [Geometric-Mean Policy Optimization](https://arxiv.org/abs/2507.20673)
4. SAPO (2025-11) [Soft Adaptive Policy Optimization](https://arxiv.org/abs/2511.20347)
5. LUSPO (2026-02) [Length-Unbiased Sequence Policy Optimization: Revealing and Controlling Response Length Variation in RLVR](https://arxiv.org/abs/2602.05261)

## 分组三：样本筛选与 prompt/group 级更新强度控制

这一组不再把所有 rollout 平等地送入 GRPO 更新，而是先判断哪些 prompt 值得大步学、哪些样本值得保留、哪些失败轨迹值得纠错后再学。它们动的不是 surrogate 主体，而是梯度入口。

1. SEED-GRPO (2025-05) [SEED-GRPO: Semantic Entropy Enhanced GRPO for Uncertainty-Aware Policy Optimization](https://arxiv.org/abs/2505.12346)
2. EDGE-GRPO (2025-07) [EDGE-GRPO: Entropy-Driven GRPO with Guided Error Correction for Advantage Diversity](https://arxiv.org/abs/2507.21848)
3. GFPO (2025-08) [Sample More to Think Less: Group Filtered Policy Optimization for Concise Reasoning](https://arxiv.org/abs/2508.09726)

## 分组四：reward-level 方法，而不是纯 GRPO loss 变体

这一组最典型的是 GAPO。它不直接改 GRPO 的 surrogate，而是改 reward 的定义对象，使 reward 不再只依赖单个回答，而是依赖同一组回答的整体分布。

1. GAPO (2025-11) [Group-Aware Reinforcement Learning for Output Diversity in Large Language Models](https://arxiv.org/abs/2511.12596)

从 TRL 的角度看，这些方法的支持情况也可以顺手标出来：

1. 原生命名支持：`DAPO`、`Dr.GRPO`、`CISPO`、`SAPO`、`LUSPO`。
2. 配置组合支持：`GSPO`，关键开关是 `importance_sampling_level="sequence"`。
3. 实验性支持：`GFPO`，位于 `trl.experimental.gfpo`。
4. 需要自定义扩展：`BNPO`、`GMPO`、`SEED-GRPO`、`EDGE-GRPO`、`GAPO`。

后面的讨论都围绕四个问题展开：

1. 这个方法具体在修什么训练症状；
2. 它改的是 advantage、归一化、ratio、mask 还是 reward 本身；
3. 它与相邻方法的边界在哪里；
4. 如果要在 TRL 中复现，最小改动点是什么。

## 1. 基线 GRPO

在讨论这些变体之前，先把基线 GRPO 写清楚。后面的很多方法看起来名字不同，实际只是对基线公式的某一部分做了替换。

设 prompt 为 $q$。旧策略 $\pi_{\theta_{\mathrm{old}}}$ 针对同一个 $q$ 采样出 $G$ 个回答 $o_1,\dots,o_G$。第 $i$ 个回答的序列级 reward 记为 $R_i = R(q,o_i)$，长度记为 $|o_i|$。

### 1.1 组相对优势

GRPO 与 PPO 的一个关键差别，是它不再显式训练独立的 critic，而是用组内相对比较来构造 advantage。最常见的写法是

$$
\hat A_i = \frac{R_i-\mu_G}{\sigma_G+\varepsilon},
\qquad
\mu_G = \frac{1}{G}\sum_{j=1}^G R_j,
\qquad
\sigma_G = \sqrt{\frac{1}{G}\sum_{j=1}^G (R_j-\mu_G)^2}.
$$

它的直觉并不复杂：同一 prompt 下，模型同时生成多个候选，然后只关心组内谁相对更好，而不是去拟合一个额外的值函数。

如果只做去均值、不做标准差缩放，则变成

$$
\hat A_i = R_i - \mu_G.
$$

这两种写法后面都会出现。是否除以组内标准差，直接关系到不同 prompt 的更新强度能否保持一致。

### 1.2 token-level clipped surrogate

对第 $i$ 个回答第 $t$ 个 token，定义 importance ratio

$$
r_{i,t}(\theta)=\frac{\pi_\theta(o_{i,t}\mid q,o_{i,<t})}{\pi_{\theta_{\mathrm{old}}}(o_{i,t}\mid q,o_{i,<t})}.
$$

则 token 级的 clipped surrogate 可以写成

$$
\ell_{i,t}(\theta)=\min\!\left(
r_{i,t}(\theta)\hat A_i,
\operatorname{clip}\!\left(r_{i,t}(\theta),1-\epsilon_{\mathrm{low}},1+\epsilon_{\mathrm{high}}\right)\hat A_i
\right).
$$

若再加入参考策略正则，则总目标可写为

$$
\mathcal{L}_{\mathrm{GRPO}}(\theta)
= -\frac{1}{\operatorname{Norm}}\sum_{i=1}^{G}\sum_{t=1}^{|o_i|} m_{i,t}\,\ell_{i,t}(\theta)
+ \beta\sum_{i=1}^{G}\sum_{t=1}^{|o_i|} m_{i,t}
D_{\mathrm{KL}}\!\left(\pi_\theta(\cdot\mid q,o_{i,<t})\Vert\pi_{\mathrm{ref}}(\cdot\mid q,o_{i,<t})\right).
$$

这里把 completion mask $m_{i,t}\in\{0,1\}$ 单独写出来，是因为后面不少工作并不是改 loss 主体，而是改哪些 token 被计入损失。

### 1.3 后续变体主要改三类对象

把上面的公式当作母式，后面的大多数工作都可以归结为三类改动。

1. 归一化与 advantage：`Norm` 是按样本、按有效 token、按常数，还是先把 reward 重参数化再送进 surrogate。
2. ratio 与 surrogate 形状：ratio 是 token-level 还是 sequence-level；clip 是硬裁剪、软门控，还是改 importance weight 本身。
3. 样本入口：哪些 rollout、哪些 token、哪些 prompt 进入梯度。

这个视角的好处是，能把很多看起来分散的工作放到统一坐标里看。比如 DAPO 和 Dr.GRPO 的差别，主要在归一化与长度解耦；GSPO、GMPO、LUSPO 的差别，主要在 ratio 粒度与长度补偿；GFPO、SEED-GRPO 的差别，则主要在梯度入口。

### 1.4 为什么长度偏置会成为主线问题

在 reasoning RL 场景里，GRPO 暴露得最早、也最稳定的问题之一，就是长度与更新强度耦合。一个直接后果是：即便 reward 改善幅度相同，长回答和短回答得到的梯度也可能系统性不同。

这个问题大致有两种表现：

1. 长回答在某些聚合方式下被平均得过弱，导致有效学习信号被稀释；
2. 某些 sequence-level 写法虽然避免了 token 级稀释，但又会把长度效应换一种形式带回来。

因此 DAPO、Dr.GRPO、GSPO、LUSPO 这些工作虽然公式不同，但都可以放在“长度偏置如何进入更新”这条主线上理解。

### 1.5 TRL 里对应的几个关键开关

从工程实现看，TRL 已经把不少关键变动暴露成配置项：

1. `loss_type`：控制 loss 聚合方式以及 surrogate 变体。
2. `importance_sampling_level`：控制 ratio 是在 token 级还是 sequence 级计算。
3. `scale_rewards`：控制是否做组标准化。
4. `mask_truncated_completions`：控制截断 completion 是否参与 loss。
5. `GFPOTrainer` 与 `group_filter_func`：控制 group filtering。

也就是说，这一章后面很多方法即便没有一键开关，通常也都能在同一个 trainer 骨架上找到明确的最小改动点。

In [ ]:
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer
from trl.rewards import think_format_reward, get_soft_overlong_punishment

SYSTEM_PROMPT = (
    "A conversation between user and assistant. The user asks a question, and the assistant first thinks, "
    "then answers inside <think></think><answer></answer>."
)

def build_demo_dataset():
    examples = [
        {
            "prompt": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": "计算 17 * 19。"},
            ],
            "ground_truth": "323",
        },
        {
            "prompt": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": "若 x + 5 = 11，求 x。"},
            ],
            "ground_truth": "6",
        },
    ]
    return Dataset.from_list(examples)

demo_dataset = build_demo_dataset()

def boxed_accuracy_reward(completions, ground_truth=None, **kwargs):
    """一个教学用 reward：检查 <answer>...</answer> 中的文本是否等于 ground_truth。"""
    rewards = []
    for completion, answer in zip(completions, ground_truth):
        if isinstance(completion, list):
            content = completion[0]["content"]
        else:
            content = completion
        left = content.find("<answer>")
        right = content.find("</answer>")
        pred = content[left + 8:right].strip() if left != -1 and right != -1 else ""
        rewards.append(1.0 if pred == answer else 0.0)
    return rewards

soft_overlong_reward = get_soft_overlong_punishment(
    max_completion_len=512,
    soft_punish_cache=128,
)

# 在真实项目中，通常还会叠加格式奖励、正确率奖励、长度惩罚等多个 reward。
reward_funcs = [think_format_reward, boxed_accuracy_reward, soft_overlong_reward]

## 2. DAPO、Dr.GRPO 与 BNPO

这一组都在处理 reward 进入 loss 之前的那一层，但三者的关注点并不相同。`DAPO` 主要处理长链路推理训练时的聚合与训练配方问题，`Dr.GRPO` 直接追问长度偏置是否来自分母设计，`BNPO` 则改写了 advantage 的形状。

### 2.1 DAPO：它首先是一套长 CoT RL recipe

DAPO 很容易被误读成“改了一个分母”。这并不准确。它真正处理的是长链路推理训练里一组同时出现的现象：长回答里的梯度稀释、过早截断造成的奖励噪声、组内全对或全错导致的零梯度，以及训练后期的探索不足。

因此 DAPO 的价值不只在一个 loss 公式，而在它把 `Clip-Higher`、`Dynamic Sampling`、token-level loss 聚合、以及 overlong shaping 组合成了一条相对完整的训练路径。

### 2.2 DAPO 的 loss 应该怎么理解

记有效 token mask 为 $m_{i,t}$，则 DAPO 的核心目标可以写成

$$
\mathcal{L}_{\mathrm{DAPO}}(\theta)
= -\frac{1}{\sum_{i=1}^{G}\sum_{t=1}^{|o_i|} m_{i,t}}
\sum_{i=1}^{G}\sum_{t=1}^{|o_i|} m_{i,t}\,\ell_{i,t}(\theta).
$$

它与基线 GRPO 的主要差别，不在 surrogate 主体，而在归一化方式。这里不是先对每个回答内部做平均，再对样本做平均，而是先把所有有效 token 统一摊平，再做一次全局归一化。

这样做的直接作用是减少长回答中的 token 被重复平均后信号变弱的问题。对于长 CoT 训练，这个差异往往会比单纯的超参数调整更明显。

### 2.3 DAPO 之所以有效，靠的不是单个公式

如果只把 DAPO 理解成 token-level normalization，很容易低估它。它起作用的原因更接近于把几个彼此耦合的问题拆开处理：

1. `Clip-Higher` 给正向探索留下更宽的上界；
2. `Dynamic Sampling` 降低大量零梯度 prompt 对训练效率的影响；
3. `mask_truncated_completions=True` 避免把截断样本简单当成失败样本；
4. soft overlong punishment 把“回答过长”和“回答错误”区分开来。

因此从方法定位上看，DAPO 更像一套稳定的工程 recipe，而不是单点理论修补。

### 2.4 DAPO 的公开评审信息说明了什么

DAPO 在公开论坛上的结果是 NeurIPS 2025 poster。结合已保存的 OpenReview 页面，可以看到一个比较稳定的评价：

1. 评审普遍认可它的系统性和工程价值，尤其是开源实现、训练细节和消融设计比较完整；
2. 主要保留意见不在效果，而在创新边界，有些 reviewer 认为多个改动放在一起有效，但单个部件的新意并不完全均衡；
3. 还有一类质疑集中在泛化范围，即这套配方是否主要服务于长数学推理，而不是一般性的 RLVR 场景。

这类评价基本符合 DAPO 的实际定位。它的价值主要在可复用训练路径，而不是在某个单独公式上建立全新的理论范式。

### 2.5 Dr.GRPO：长度偏置可能来自分母，而不只来自聚合粒度

如果说 DAPO 解决的是“长 CoT RL 怎么更稳地跑起来”，那么 Dr.GRPO 讨论的是另一个更基础的问题：即便已经做 token-level 聚合，loss scale 是否仍然会和实际回答长度耦合。

它给出的修正方式很直接，就是把归一化分母改成常数 $L$，通常取 `max_completion_length`：

$$
\mathcal{L}_{\mathrm{Dr.GRPO}}(\theta)
= -\frac{1}{GL}\sum_{i=1}^{G}\sum_{t=1}^{|o_i|} m_{i,t}\,\ell_{i,t}(\theta).
$$

这样做的含义是，不再让不同长度的回答通过分母差异影响整体更新尺度。长度仍然存在于 token 求和里，但不再通过样本内归一化再次进入 loss。

Dr.GRPO 通常还配合另一项建议：只做去均值，不做组标准差缩放，即

$$
\hat A_i = R_i - \mu_G.
$$

原因是组标准差本身也会把题目难度和组内 reward 波动重新带回更新强度。

### 2.6 Dr.GRPO 的公开评审信息说明了什么

Dr.GRPO 对应的公开论坛标题是 Understanding R1-Zero-Like Training: A Critical Perspective，最终被 COLM 2025 接收。公开评审里能看到比较典型的分歧：

1. 支持意见认为论文把长度偏置问题讲得足够明确，尤其把一些经验观察整理成了更可检验的理论批评；
2. 保留意见主要集中在证据强度，部分 reviewer 希望看到更细的 ablation 和更严格的统计支持；
3. 决策意见总体是正面的，但也指出论文叙述上有些地方仍偏粗糙，需要更凝练的表达。

这和方法本身的性质是一致的。Dr.GRPO 最有价值的地方是提出了一种批评框架，迫使我们重新检查原本默认接受的归一化方式。

### 2.7 BNPO：reward 不一定非要线性变成 advantage

BNPO 与前两者不在同一条线上。它不再集中讨论长度，而是追问另一个问题：reward 变成 advantage 时，是否必须采用线性中心化或标准化。

BNPO 的做法是先按 reward 排序，再把排序位置映射到 Beta 分布的分位数。若组内共有 $G$ 个回答，第 $i$ 个样本的分位位置定义为

$$
\nu_i = \frac{\operatorname{rank}(i)-0.5}{G},
\qquad \nu_i \in (0,1),
$$

则 advantage 写成

$$
A_i^{\mathrm{BNPO}} = F_{\mathrm{Beta}(\alpha,\beta)}^{-1}(\nu_i).
$$

再把这个 advantage 送入标准 surrogate：

$$
\mathcal{L}_{\mathrm{BNPO}}(\theta)
= -\frac{1}{\operatorname{Norm}}\sum_{i=1}^{G}\sum_{t=1}^{|o_i|}
\min\!\left(
r_{i,t}(\theta)A_i^{\mathrm{BNPO}},
\operatorname{clip}\!\left(r_{i,t}(\theta),1-\epsilon,1+\epsilon\right)A_i^{\mathrm{BNPO}}
\right).
$$

从这个角度看，BNPO 的核心不是重新定义 trust region，而是重新定义 reward 差异在梯度里应该被放大成什么形状。

BNPO在openreview中的评审争议比较集中，可以提炼出几个核心点：

1. 支持意见认可它试图用统一的参数化视角讨论 advantage shaping，这一点在组织理论问题上是有价值的；
2. 主要质疑在于理论语义是否站得住，即这种由排序和分布拟合得到的 advantage，是否仍然与原始 policy gradient 目标保持足够清晰的对应关系；
3. 经验收益被认为是存在的，但部分 reviewer 认为收益与额外复杂度之间的平衡还不够明确；
4. 最终结果是 ICLR 2026 withdraw/reject，说明这条路线提出的问题值得重视，但当前答案还没有得到充分共识.


In [ ]:
# DAPO: 直接使用 GRPOTrainer + loss_type="dapo"
dapo_args = GRPOConfig(
    output_dir="./outputs/grpo-dapo",
    learning_rate=1e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=8,
    max_completion_length=512,
    loss_type="dapo",
    mask_truncated_completions=True,
    epsilon=0.2,
    epsilon_high=0.28,
    beta=0.0,
    logging_steps=10,
    report_to="none",
)

dapo_trainer = GRPOTrainer(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    reward_funcs=reward_funcs,
    args=dapo_args,
    train_dataset=demo_dataset,
)

# Dr.GRPO: 只换 loss_type 和 reward scaling 策略
drgrpo_args = GRPOConfig(
    output_dir="./outputs/grpo-drgrpo",
    learning_rate=1e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=8,
    max_completion_length=512,
    loss_type="dr_grpo",
    scale_rewards="none",
    beta=0.0,
    logging_steps=10,
    report_to="none",
)

drgrpo_trainer = GRPOTrainer(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    reward_funcs=reward_funcs,
    args=drgrpo_args,
    train_dataset=demo_dataset,
)

# BNPO(Beta Normalization Policy Optimization) 不是 TRL 的 loss_type="bnpo"。
# 下面给的是“论文版 BNPO”的示意实现：保留 GRPOTrainer 主体，只在 advantages 生成后做 rank->Beta quantile 映射。
import torch


def beta_rank_advantages(reward_groups, alpha=2.0, beta=2.0):
    """示意实现：把每组 reward 排序后映射到 Beta 分位数。"""
    mapped_groups = []
    for rewards in reward_groups:
        rewards = torch.as_tensor(rewards, dtype=torch.float32)
        group_size = rewards.numel()
        order = torch.argsort(rewards)
        ranks = torch.empty_like(order, dtype=torch.float32)
        ranks[order] = torch.arange(1, group_size + 1, dtype=torch.float32)
        quantiles = (ranks - 0.5) / group_size

        beta_dist = torch.distributions.Beta(
            torch.tensor(float(alpha)),
            torch.tensor(float(beta)),
        )
        if not hasattr(beta_dist, "icdf"):
            raise NotImplementedError("当前环境没有可用的 Beta icdf；可换成 scipy.stats.beta.ppf 或数值求逆。")
        mapped_groups.append(beta_dist.icdf(quantiles))
    return mapped_groups


class BetaBNPOTrainer(GRPOTrainer):
    """示意实现：论文版 BNPO 的最小改动点是 advantages，而不是 loss_type。"""

    def __init__(self, *args, bnpo_alpha=2.0, bnpo_beta=2.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.bnpo_alpha = bnpo_alpha
        self.bnpo_beta = bnpo_beta

    def _generate_and_score_completions(self, inputs):
        outputs = super()._generate_and_score_completions(inputs)

        num_generations = self.num_generations if self.model.training else self.num_generations_eval
        if num_generations is None or num_generations <= 1:
            return outputs

        advantages = outputs["advantages"].view(-1, num_generations)
        mapped = beta_rank_advantages(
            advantages.detach().cpu(),
            alpha=self.bnpo_alpha,
            beta=self.bnpo_beta,
        )
        outputs["advantages"] = torch.stack(mapped).to(advantages.device).reshape(-1)
        return outputs


bnpo_trainer = BetaBNPOTrainer(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    reward_funcs=reward_funcs,
    args=dapo_args,
    train_dataset=demo_dataset,
    bnpo_alpha=2.0,
    bnpo_beta=2.0,
)

# 真正训练时执行：
# dapo_trainer.train()
# drgrpo_trainer.train()
# bnpo_trainer.train()

## 3. CISPO、GSPO、GMPO、SAPO 与 LUSPO

这一组默认 reward 和 advantage 的构造已经给定，主要讨论 policy shift 应该如何表述，以及 surrogate 应该如何施加约束。从技术层次上看，它们处理的是 importance ratio 与 trust region 的设计问题。

### 3.1 CISPO：先裁 importance weight，再保留梯度信号

PPO/GRPO 的标准做法是直接裁 surrogate。CISPO 的出发点不同，它认为很多 token 在 ratio 超界之后就直接被截断，会损失可用的学习信号，尤其是在长回答和高方差训练阶段。

对第 $i$ 个回答第 $t$ 个 token，定义 importance weight

$$
w_{i,t}(\theta)=\frac{\pi_\theta(o_{i,t}\mid q,o_{i,<t})}{\pi_{\theta_{\mathrm{old}}}(o_{i,t}\mid q,o_{i,<t})}.
$$

CISPO 先做裁剪

$$
\tilde w_{i,t}(\theta)=\operatorname{clip}\!\left(w_{i,t}(\theta),1-\epsilon_{\mathrm{low}},1+\epsilon_{\mathrm{high}}\right),
$$

再构造目标

$$
J_{\mathrm{CISPO}}(\theta)
= \mathbb{E}\left[
\frac{1}{\sum_i |o_i|}
\sum_i\sum_t
\operatorname{sg}(\tilde w_{i,t}(\theta))\,\hat A_{i,t}
\log \pi_\theta(o_{i,t}\mid q,o_{i,<t})
\right].
$$

它与 GRPO 的实质差异，在于裁剪发生的位置不同。GRPO 裁的是 surrogate，CISPO 裁的是 importance weight，本质上是在尽量保留 REINFORCE 风格的梯度形态。

### 3.2 CISPO 的方法定位

CISPO 在 TRL 中已经有原生支持：`loss_type="cispo"`。这说明它不是停留在概念层面的替代写法，而是已经被整理成稳定实现。

从实验设计上看，CISPO 更适合回答下面这个问题：在相同 trust region 约束下，是否存在一种写法，能够比标准 clipped surrogate 保留更多有效梯度。如果训练成本敏感、或者训练经常出现更新步利用率偏低，CISPO 是值得优先做的对照项。

### 3.3 GSPO：ratio 不再按 token 看，而是按 sequence 看

GSPO 的出发点是，很多 reasoning RL 任务的 reward 天然是序列级的，因此把 ratio 也提升到 sequence 级是一个自然想法。

它定义 sequence-level ratio 为

$$
s_i(\theta)
=\left(\frac{\pi_\theta(o_i\mid q)}{\pi_{\theta_{\mathrm{old}}}(o_i\mid q)}\right)^{1/|o_i|}
=\exp\!\left(\frac{1}{|o_i|}\sum_{t=1}^{|o_i|}\log r_{i,t}(\theta)\right).
$$

在此基础上，对应的 surrogate 可以写成

$$
\mathcal{L}_{\mathrm{GSPO}}(\theta)
= -\frac{1}{G}\sum_{i=1}^{G}
\min\!\left(
s_i(\theta)\hat A_i,
\operatorname{clip}\!\left(s_i(\theta),1-\epsilon_{\mathrm{low}},1+\epsilon_{\mathrm{high}}\right)\hat A_i
\right).
$$

这里的几何平均不是形式上的修饰，而是为了抵消 sequence likelihood 乘法结构天然带来的长度膨胀效应。如果直接使用整句概率比，长回答的尺度会变得很难控制。

在 TRL 中，GSPO 的主结构可以通过 `importance_sampling_level="sequence"` 组合出来，因此它属于“配置组合支持”，而不是完全需要另写 trainer。

### 3.4 GMPO：仍是 sequence-level，但更强调几何均值与更宽 clip 的配套关系

GMPO 与 GSPO 关系很近，但关注点更偏训练动力学。它强调 sequence-level 更新如果仍沿用过窄的裁剪区间，往往会过于保守，使几何均值 ratio 的潜力发挥不出来。

若记

$$
g_i(\theta)=\left(\prod_{t=1}^{|o_i|}r_{i,t}(\theta)\right)^{1/|o_i|},
$$

则其目标可写成

$$
\mathcal{L}_{\mathrm{GMPO}}(\theta)
= -\frac{1}{G}\sum_{i=1}^{G}
\min\!\left(
g_i(\theta)\hat A_i,
\operatorname{clip}\!\left(g_i(\theta),1-\epsilon_\ell,1+\epsilon_h\right)\hat A_i
\right).
$$

从形式上看，它与 GSPO 非常接近；差别主要在它更强调几何均值 ratio 与裁剪上界需要联动设计。TRL 目前没有单独的 `gmpo` 开关，因此更适合视为 sequence-level 路线上的自定义扩展。

### 3.5 SAPO：把 hard clip 改成连续衰减的 soft gate

SAPO 处理的是另一个问题。标准 clip 在边界处是非光滑的，更新一旦碰到裁剪边缘，梯度行为会发生突变。SAPO 试图用软门控替换这种刚性边界。

设

$$
g_+(r)=\sigma\!\left(\frac{r-1}{\tau_+}\right),
\qquad
g_-(r)=\sigma\!\left(\frac{1-r}{\tau_-}\right),
$$

则可以写出它的目标

$$
\mathcal{L}_{\mathrm{SAPO}}(\theta)
= -\sum_{i,t}
\big[\mathbf{1}(\hat A_i\ge 0)g_+(r_{i,t}) + \mathbf{1}(\hat A_i<0)g_-(r_{i,t})\big]
r_{i,t}(\theta)\hat A_i.
$$

它不改变 ratio 粒度，也不改变 advantage 语义，而是把 trust region 从“硬边界”改成“连续衰减”。如果训练对边界附近的震荡比较敏感，SAPO 往往比 hard clip 更容易调。

TRL 已经提供 `loss_type="sapo"`，并暴露了 `sapo_temperature_pos`、`sapo_temperature_neg` 两个温度参数。

### 3.6 LUSPO：sequence-level 路线上的长度无偏修正

GSPO 把 ratio 提升到了 sequence 级，但这并不意味着长度问题自动消失。LUSPO 指出，几何平均 ratio 仍会系统性压缩长回答的更新幅度，因此需要显式补偿长度。

仍记

$$
s_i(\theta)=\left(\prod_{t=1}^{|o_i|}r_{i,t}(\theta)\right)^{1/|o_i|},
$$

则 LUSPO 的目标写成

$$
\mathcal{L}_{\mathrm{LUSPO}}(\theta)
= -\frac{1}{G}\sum_{i=1}^{G}|o_i|
\min\!\left(
s_i(\theta)\hat A_i,
\operatorname{clip}\!\left(s_i(\theta),1-\epsilon,1+\epsilon\right)\hat A_i
\right).
$$

这里显式乘上 $|o_i|$，目的就是把 sequence-level 几何平均中被压缩掉的长度尺度补回来。也可以说，LUSPO 不是否定 GSPO，而是在 sequence-level 路线上进一步把长度无偏性问题写清楚。

在 TRL 中，它已经有原生命名支持：`loss_type="luspo"`，同时需要配合 `importance_sampling_level="sequence"`。

### 3.7 这一组方法如何安排对照实验

如果按研究问题来排顺序，一个自然的实验路径是：

1. 用 `GSPO` 作为 sequence-level 路线的起点；
2. 用 `LUSPO` 检查长度补偿是否带来稳定收益；
3. 用 `CISPO` 检查裁剪位置改变后，是否能提高有效梯度利用率；
4. 用 `SAPO` 检查 soft gate 是否优于 hard clip；
5. 把 `GMPO` 放在 sequence-level 路线已经确认有效之后，再分析更宽 clip 是否真的带来探索收益。

这一组的优点是改动位置相对单纯，因此比 recipe 型工作更适合做严格的消融和归因分析。

In [ ]:
# CISPO: TRL 已原生支持
cispo_args = GRPOConfig(
    output_dir="./outputs/grpo-cispo",
    learning_rate=1e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=8,
    max_completion_length=512,
    loss_type="cispo",
    epsilon=0.2,
    epsilon_high=5.0,
    beta=0.0,
    logging_steps=10,
    report_to="none",
)

cispo_trainer = GRPOTrainer(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    reward_funcs=reward_funcs,
    args=cispo_args,
    train_dataset=demo_dataset,
)

# GSPO: 通过 importance_sampling_level="sequence" 组合支持
gspo_args = GRPOConfig(
    output_dir="./outputs/grpo-gspo",
    learning_rate=1e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=8,
    max_completion_length=512,
    importance_sampling_level="sequence",
    loss_type="grpo",
    epsilon=0.2,
    epsilon_high=0.28,
    beta=0.0,
    logging_steps=10,
    report_to="none",
)

gspo_trainer = GRPOTrainer(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    reward_funcs=reward_funcs,
    args=gspo_args,
    train_dataset=demo_dataset,
)

# SAPO: TRL 已原生支持软门控 loss
sapo_args = GRPOConfig(
    output_dir="./outputs/grpo-sapo",
    learning_rate=1e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=8,
    max_completion_length=512,
    loss_type="sapo",
    sapo_temperature_pos=1.0,
    sapo_temperature_neg=1.05,
    beta=0.0,
    logging_steps=10,
    report_to="none",
)

sapo_trainer = GRPOTrainer(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    reward_funcs=reward_funcs,
    args=sapo_args,
    train_dataset=demo_dataset,
)

# LUSPO: TRL 已原生支持，但必须配 sequence-level importance sampling
luspo_args = GRPOConfig(
    output_dir="./outputs/grpo-luspo",
    learning_rate=1e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=8,
    max_completion_length=512,
    importance_sampling_level="sequence",
    loss_type="luspo",
    beta=0.0,
    logging_steps=10,
    report_to="none",
)

luspo_trainer = GRPOTrainer(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    reward_funcs=reward_funcs,
    args=luspo_args,
    train_dataset=demo_dataset,
)

# GMPO: TRL 没有单独的 loss_type="gmpo"。
# 下面先给一个“组合近似版”，它复用 GSPO 的 sequence ratio，再把 clip 上界放宽。
gmpo_like_args = GRPOConfig(
    output_dir="./outputs/grpo-gmpo-like",
    learning_rate=1e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=8,
    max_completion_length=512,
    importance_sampling_level="sequence",
    loss_type="grpo",
    epsilon=0.2,
    epsilon_high=0.4,
    beta=0.0,
    logging_steps=10,
    report_to="none",
)


class GMPOTrainer(GRPOTrainer):
    """示意实现：若要严格复现 GMPO，可在 sequence ratio 与 clipping 统计上做定制。"""

    @staticmethod
    def geometric_mean_ratio(token_logprobs, old_token_logprobs, completion_mask):
        token_log_ratio = (token_logprobs - old_token_logprobs) * completion_mask
        valid_lengths = completion_mask.sum(dim=-1).clamp_min(1)
        seq_log_ratio = token_log_ratio.sum(dim=-1) / valid_lengths
        return seq_log_ratio.exp()

    # 在真实实现中，可进一步重写 _compute_loss 或相关 ratio 聚合逻辑，
    # 把 geometric_mean_ratio 明确地接入 surrogate 与日志统计。


# 真正训练时执行：
cispo_trainer.train()
gspo_trainer.train()
sapo_trainer.train()
luspo_trainer.train()

## 4. SEED-GRPO、EDGE-GRPO 与 GFPO

这一组方法的共同点，是不再默认所有 rollout 都应该以同样方式进入 GRPO 更新。它们处理的不是 surrogate 主体，而是梯度入口，因此和前面的 DAPO、GSPO 一类方法通常可以正交组合。

### 4.1 SEED-GRPO：按 prompt 不确定性调节更新强度

SEED-GRPO 的直觉是，同样是 reward 有差异的两组 rollout，如果一组答案语义高度集中，另一组高度分散，那么这两组样本未必应该有相同的学习步长。它试图把这种差异写成一个 prompt-level 的重加权因子。

设对同一个 prompt $q$ 采样得到 $G$ 个回答，并被聚成 $K$ 个语义簇 $C_1,\dots,C_K$，记

$$
p_k=\frac{|C_k|}{G},\qquad k=1,\dots,K,
$$

则 semantic entropy 定义为

$$
H_{\mathrm{sem}}(q)=-\sum_{k=1}^{K}p_k\log p_k.
$$

SEED-GRPO 不改 surrogate，而是把 advantage 重新缩放为

$$
\tilde A_i = w(q)\hat A_i,
\qquad
w(q)=1-\alpha\frac{H_{\mathrm{sem}}(q)}{\log G}.
$$

因此它真正做的是 prompt-level update modulation。高熵 prompt 的更新会被压小，低熵 prompt 的更新则更接近原始 GRPO。

### 4.2 SEED-GRPO 的公开评审信息说明了什么

SEED-GRPO 的公开论坛结论很有代表性。根据本地保存的 OpenReview 页面，可以概括为：

1. 评审承认它抓住了一个真实问题，即不同 prompt 的 rollout 可靠性并不相同；
2. 主要质疑集中在“semantic entropy”这个命名是否过强，因为实际实现更接近 final-answer clustering，而不是真正建模完整 reasoning 过程的语义熵；
3. 还有一类保留意见指向新意边界，即它与已有 uncertainty-aware、consistency-aware 路线相比，到底新增了多少东西；
4. 最终结果是 ICLR 2026 reject，反映的不是结果无效，而是叙事和 claim 比实现本身走得更远。

因此在使用时，更稳妥的理解方式是把它看成一种 prompt-level uncertainty reweighting，而不是一个已经完全解决语义不确定性建模的问题。

### 4.3 EDGE-GRPO：先分析样本，再纠错，再重写优势

EDGE-GRPO 比 SEED-GRPO 更进一步。它不满足于只对 prompt 乘一个权重，而是显式区分哪些失败轨迹值得保留、哪些可以纠正后再学、哪些应当直接排除出更新。

把它抽象写出来，可以表示为

$$
\tilde A_i = m_i\big(\hat A_i + \lambda\phi(H_i)\big),
$$

其中 $m_i$ 表示样本是否保留，$\phi(H_i)$ 是由熵或不确定性构造的附加项。更关键的是，这个式子之前往往还接了一步 Guided Error Correction，也就是先把部分失败轨迹修成更有训练价值的版本。

因此 EDGE-GRPO 更适合被理解成一个三段式系统：

1. 样本分析；
2. 错误纠正；
3. 优势重写。

它的重点不在提出一个新的标准 surrogate，而在于把 selective learning 和 corrective learning 写进 RL 流水线。

### 4.4 GFPO：先扩大候选，再只学习最值得学的部分

GFPO 针对的是另一个很实际的问题。reasoning RL 里常见的现象是回答越来越长，但增加的 token 不一定都带来有效推理。GFPO 的思路不是直接惩罚长度，而是先采样更多候选，再只保留更有学习价值的子集进入更新。

设候选组为 $\mathcal{G}(q)=\{o_1,\dots,o_G\}$，保留子集为 $S(q)\subset\mathcal{G}(q)$，定义样本掩码

$$
m_i=\mathbf{1}\{o_i\in S(q)\}.
$$

则保留子集上的统计量为

$$
\mu_S = \frac{1}{|S|}\sum_{o_j\in S}R(q,o_j),
\qquad
\sigma_S = \sqrt{\frac{1}{|S|}\sum_{o_j\in S}(R(q,o_j)-\mu_S)^2}.
$$

对应的 masked advantage 为

$$
\hat A_i^{(m)} = m_i\cdot\frac{R(q,o_i)-\mu_S}{\sigma_S+\varepsilon}.
$$

这样写以后，GFPO 的核心思想就比较明确了：不是所有 rollout 都值得参与学习，先做筛选，再谈优势归一化。

### 4.5 GFPO 的工程意义

GFPO 在 TRL 中已经有实验性实现：`trl.experimental.gfpo.GFPOTrainer`。这意味着它不是只能停留在论文概念层面的想法，而是已经有相对明确的工程入口。

从工程角度看，GFPO 的优势在于它把一个经常靠经验手动完成的动作正规化了：先多探索，再按一套可复用规则过滤样本，把有限的 token budget 用在更值得学的轨迹上。

### 4.6 这一组方法的边界

这三条线经常会被混在一起讲，但它们改动的层次并不相同：

1. `SEED-GRPO` 改的是 prompt-level 更新强度；
2. `EDGE-GRPO` 改的是样本处理流水线，并把纠错纳入训练；
3. `GFPO` 改的是进入梯度的样本集合。

正因为层次不同，它们通常可以与 DAPO、GSPO、LUSPO 这类 optimizer 内部方法组合，而不是简单互斥。

### 4.7 这一组方法在实验中怎么排顺序

如果从实现难度和收益确定性来排，通常可以按下面顺序推进：

1. 先试 `GFPO`，因为它的目标最直接，实现入口也最清楚；
2. 再试 `SEED-GRPO`，检查 uncertainty-aware reweighting 是否改善训练稳定性；
3. 最后再看 `EDGE-GRPO`，因为它的系统性最强，但实现复杂度也最高。

这样安排的好处是，可以先把样本选择是否有收益这件事验证清楚，再逐步引入更重的分析和纠错模块。

In [ ]:
# GFPO: TRL 提供实验性 trainer
from trl.experimental.gfpo import GFPOConfig, GFPOTrainer
import math
import re
import torch


class ConciseReasoningFilter:
    """示例：优先保留 reward 高、且回答更短的样本。"""

    def __call__(self, group_completions, group_rewards, **kwargs):
        group_scores = []
        for completions, rewards in zip(group_completions, group_rewards):
            scores = []
            for completion, reward in zip(completions, rewards):
                if isinstance(completion, list):
                    content = completion[0]["content"]
                else:
                    content = completion
                length_penalty = 0.001 * len(content)
                scores.append(float(reward) - length_penalty)
            group_scores.append(scores)
        return group_scores


gfpo_args = GFPOConfig(
    output_dir="./outputs/grpo-gfpo",
    learning_rate=1e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=8,
    num_remains_in_group=4,
    max_completion_length=512,
    loss_type="dapo",
    logging_steps=10,
    report_to="none",
)

gfpo_trainer = GFPOTrainer(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    reward_funcs=reward_funcs,
    args=gfpo_args,
    train_dataset=demo_dataset,
    group_filter_func=ConciseReasoningFilter(),
)


class SeedGRPOTrainer(GRPOTrainer):
    """示意实现：在 GRPOTrainer 上增加 prompt-level semantic entropy reweighting。"""

    def __init__(self, *args, seed_alpha=0.02, **kwargs):
        super().__init__(*args, **kwargs)
        self.seed_alpha = seed_alpha

    @staticmethod
    def _extract_final_answer(text):
        boxed = re.findall(r"\\boxed\{([^}]*)\}", text)
        if boxed:
            return boxed[-1].strip()
        answer_match = re.search(r"<answer>(.*?)</answer>", text, flags=re.S)
        if answer_match:
            return answer_match.group(1).strip()
        return None

    def _semantic_entropy_weight(self, group_completions):
        labels = []
        no_answer_count = 0
        for completion in group_completions:
            content = completion[0]["content"] if isinstance(completion, list) else completion
            answer = self._extract_final_answer(content)
            if answer is None or answer == "":
                labels.append(f"__no_answer_{no_answer_count}")
                no_answer_count += 1
            else:
                labels.append(answer)

        group_size = len(labels)
        counts = {}
        for label in labels:
            counts[label] = counts.get(label, 0) + 1

        entropy = 0.0
        for count in counts.values():
            prob = count / group_size
            entropy -= prob * math.log(prob + 1e-12)

        entropy_max = math.log(group_size) if group_size > 1 else 1.0
        weight = 1.0 - self.seed_alpha * entropy / entropy_max
        return max(weight, 0.0)

    def _generate_and_score_completions(self, inputs):
        outputs = super()._generate_and_score_completions(inputs)

        num_generations = self.num_generations if self.model.training else self.num_generations_eval
        if num_generations is None or num_generations <= 1:
            return outputs

        completions = outputs["completions"]
        advantages = outputs["advantages"].view(-1, num_generations)

        weights = []
        for start in range(0, len(completions), num_generations):
            group = completions[start:start + num_generations]
            weights.append(self._semantic_entropy_weight(group))

        weight_tensor = torch.tensor(
            weights,
            device=advantages.device,
            dtype=advantages.dtype,
        ).unsqueeze(1)

        outputs["advantages"] = (advantages * weight_tensor).reshape(-1)
        return outputs


class EDGEGRPOTrainer(GRPOTrainer):
    """示意实现：在 rollout 后接入样本分析、纠错和 entropy-driven advantage。"""

    def __init__(self, *args, edge_lambda=0.05, **kwargs):
        super().__init__(*args, **kwargs)
        self.edge_lambda = edge_lambda

    def analyze_group(self, group_completions, group_rewards):
        """返回每个样本的保留标记、纠错标记和熵奖励。"""
        analysis = []
        for completion, reward in zip(group_completions, group_rewards):
            content = completion[0]["content"] if isinstance(completion, list) else completion
            entropy_bonus = min(len(content) / 1000.0, 0.2)
            keep = reward > 0.0 or entropy_bonus > 0.08
            needs_correction = reward <= 0.0 and entropy_bonus > 0.08
            analysis.append(
                {
                    "keep": keep,
                    "needs_correction": needs_correction,
                    "entropy_bonus": entropy_bonus,
                }
            )
        return analysis

    def guided_error_correction(self, completion):
        """示意钩子：真实系统中可调用纠错模型或规则化修正器。"""
        return completion

    def _generate_and_score_completions(self, inputs):
        outputs = super()._generate_and_score_completions(inputs)

        num_generations = self.num_generations if self.model.training else self.num_generations_eval
        if num_generations is None or num_generations <= 1:
            return outputs

        completions = outputs["completions"]
        rewards = outputs["rewards"].view(-1, num_generations)
        advantages = outputs["advantages"].view(-1, num_generations)

        rewritten_groups = []
        for group_idx, start in enumerate(range(0, len(completions), num_generations)):
            group = completions[start:start + num_generations]
            group_rewards = rewards[group_idx].tolist()
            analysis = self.analyze_group(group, group_rewards)

            adjusted_advantages = advantages[group_idx].clone()
            for sample_idx, sample_info in enumerate(analysis):
                if not sample_info["keep"]:
                    adjusted_advantages[sample_idx] = 0.0
                    continue
                adjusted_advantages[sample_idx] += self.edge_lambda * sample_info["entropy_bonus"]
                if sample_info["needs_correction"]:
                    group[sample_idx] = self.guided_error_correction(group[sample_idx])
            rewritten_groups.append(adjusted_advantages)

        outputs["advantages"] = torch.stack(rewritten_groups).reshape(-1)
        return outputs


seed_args = GRPOConfig(
    output_dir="./outputs/grpo-seed",
    learning_rate=1e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=8,
    max_completion_length=512,
    loss_type="dapo",
    beta=0.0,
    logging_steps=10,
    report_to="none",
)

seed_trainer = SeedGRPOTrainer(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    reward_funcs=reward_funcs,
    args=seed_args,
    train_dataset=demo_dataset,
    seed_alpha=0.02,
)

# 真正训练时执行：
# gfpo_trainer.train()
# seed_trainer.train()
# EDGEGRPOTrainer(...).train()

## 5. GAPO：reward 的定义对象变了

这一节单独拿出来讲，是为了把边界划清楚。这里讨论的 `GAPO` 指的是 Group-Aware Reinforcement Learning for Output Diversity in Large Language Models。它与前面那些方法不在同一层，因为它改的不是 GRPO 内部的 surrogate，而是 reward 的定义对象。

### 5.1 GAPO 处理的是单样本 reward 无法表达的目标

前面各类 GRPO 变体默认一个前提：reward 首先是单样本的，也就是先给每个 rollout 一个分数，再讨论 advantage、ratio 和裁剪方式。GAPO 不接受这个前提。

它关注的场景是，同一个 prompt 可能存在多个合理答案，但模型在 RL 后容易塌缩到少数模式。此时单条回答是否正确并不能完整反映训练目标，更重要的是整组输出的分布是否过于集中。

### 5.2 group-aware reward 的基本写法

设同一 prompt 下生成一组回答

$$
S=\{o_1,o_2,\dots,o_G\}.
$$

GAPO 不再先写单样本 reward $R(q,o_i)$，而是先引入 group-level 量

$$
R_{\mathrm{group}} = R(S).
$$

在论文讨论的列表选择任务中，设有效答案集合为 $V=\{v_1,\dots,v_L\}$，对每个有效答案定义组内频率

$$
f_v(o)=\frac{\sum_{i=1}^{G}\mathbf{1}\{o_i=v\}}{\sum_{i=1}^{G}\mathbf{1}\{o_i\in V\}}.
$$

若目标分布是均匀分布 $u=1/L$，则第 $i$ 个 rollout 的 frequency-aware reward 可以写成

$$
\tilde R(o)_i=
\begin{cases}
1-\left(f_{o_i}(o)-\frac{1}{L}\right), & o_i\in V,\\
-1, & \text{otherwise}.
\end{cases}
$$

这个定义的含义是，正确答案也不是一律同奖，而是要看它在当前组里是否已经被过度重复。换句话说，GAPO 奖励的不只是“答对”，还包括“不要让少数模式垄断整组输出”。

### 5.3 GAPO 与 GRPO 的关系

GAPO 并没有替代 GRPO，而是在 GRPO 前面加了一层 group-aware reward construction：

1. 先根据同一组输出的分布性质构造 reward；
2. 再把这些 reward 交给标准 GRPO 去计算 advantage 和 loss。

因此从实现上看，它不是新的 `loss_type`，而是 custom group reward on top of GRPO。

### 5.4 为什么它需要单独归类

如果把 GAPO 混进“GRPO loss 变体”，后面的比较很容易失真。因为：

1. `DAPO`、`Dr.GRPO`、`GSPO`、`SAPO`、`LUSPO` 改的是 optimizer 内部结构；
2. `GFPO`、`SEED-GRPO`、`EDGE-GRPO` 改的是样本进入更新的方式；
3. `GAPO` 改的是 reward 的定义对象。

这三个层次是可以正交叠加的，不应该被当成同类替代关系。

### 5.5 GAPO 在 TRL 中怎么落地

TRL 里没有专门的 `GAPOTrainer`，但实现门槛并不高，因为最小改动点非常清楚：

1. 仍然使用 `GRPOTrainer`；
2. 对每个 prompt 生成一组候选；
3. 在 reward function 内部按组统计答案频率并构造 group-aware reward；
4. 再走标准 GRPO 更新。

因此 GAPO 更像 reward function 的扩展，而不是 trainer 结构的重写。对于已经有稳定 GRPO 管线的项目，这类方法往往比改 optimizer 更容易插入实验。

### 5.6 对 GAPO 的方法判断

GAPO 的主要意义，在于提醒我们有些后训练目标本来就不是单样本评分问题，而是分布控制问题。只要任务允许多解，单靠温度、采样和拒绝采样往往不足以稳定维持多样性，多样性本身需要进入 reward。

从这个意义上说，GAPO 更接近 reward 建模方法，而不是 optimizer 方法。把这一点看清楚，它在本章中的位置就很明确。

In [ ]:
from collections import Counter


# GAPO: 当前没有官方 GAPOTrainer，但可以把 group-aware reward 直接接到 GRPOTrainer 上。
# 下面给出一个列表选择任务的 frequency-aware reward 示意实现。
GAPO_NUM_GENERATIONS = 8


def extract_choice(text, valid_answers):
    """示意：从 completion 中抽取最终选项。"""
    for answer in valid_answers:
        if answer in text:
            return answer
    return None


class GroupAwareReward:
    def __init__(self, num_generations):
        self.num_generations = num_generations

    def __call__(self, completions, valid_answers=None, target_distribution=None, **kwargs):
        rewards = []
        target_distribution = target_distribution or []

        for start in range(0, len(completions), self.num_generations):
            group = completions[start:start + self.num_generations]
            group_valid_answers = valid_answers[start:start + self.num_generations]
            group_targets = target_distribution[start:start + self.num_generations]

            answers = group_valid_answers[0]
            targets = group_targets[0] if group_targets else None
            if targets is None:
                targets = {answer: 1.0 / len(answers) for answer in answers}

            parsed = []
            for completion in group:
                content = completion[0]["content"] if isinstance(completion, list) else completion
                parsed.append(extract_choice(content, answers))

            valid_only = [answer for answer in parsed if answer is not None]
            if not valid_only:
                rewards.extend([-1.0] * len(group))
                continue

            counts = Counter(valid_only)
            total_valid = sum(counts.values())

            for parsed_answer in parsed:
                if parsed_answer is None:
                    rewards.append(-1.0)
                    continue
                empirical_freq = counts[parsed_answer] / total_valid
                target_freq = targets.get(parsed_answer, 0.0)
                rewards.append(1.0 - (empirical_freq - target_freq))

        return rewards


def build_gapo_dataset():
    examples = [
        {
            "prompt": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": "从[A, B, C, D]中选一个你最喜欢的选项，并解释原因。"},
            ],
            "valid_answers": ["A", "B", "C", "D"],
            "target_distribution": {"A": 0.25, "B": 0.25, "C": 0.25, "D": 0.25},
        },
        {
            "prompt": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": "从[红色, 蓝色, 绿色, 黄色]中选一个颜色并给出一句文案。"},
            ],
            "valid_answers": ["红色", "蓝色", "绿色", "黄色"],
            "target_distribution": {"红色": 0.25, "蓝色": 0.25, "绿色": 0.25, "黄色": 0.25},
        },
    ]
    return Dataset.from_list(examples)


gapo_dataset = build_gapo_dataset()
gapo_reward = GroupAwareReward(num_generations=GAPO_NUM_GENERATIONS)

gapo_args = GRPOConfig(
    output_dir="./outputs/grpo-gapo",
    learning_rate=1e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=GAPO_NUM_GENERATIONS,
    max_completion_length=256,
    loss_type="dapo",
    beta=0.0,
    logging_steps=10,
    report_to="none",
)

gapo_trainer = GRPOTrainer(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    reward_funcs=gapo_reward,
    args=gapo_args,
    train_dataset=gapo_dataset,
)

# 如果任务不是均匀目标分布，而是希望贴近某个给定先验分布，
# 只需要把 target_distribution 换成对应的字典即可。

# 真正训练时执行：
# gapo_trainer.train()

## 6. 最后的归纳：先看方法改了哪一层，再看结果表

把这一章的方法放在一起看，最重要的不是记住多少缩写，而是把改动层次分清楚。只要层次不清，实验表里的比较就很难解释。

### 6.1 按修改层次重新整理

第一类，改的是 advantage 与归一化，也就是 reward 进入 loss 之前的形状和尺度。这一类包括 `DAPO`、`Dr.GRPO`、`BNPO`。

第二类，改的是 importance ratio 与 surrogate 结构，也就是 policy shift 应该按 token 看还是按 sequence 看，trust region 应该怎样施加。这一类包括 `CISPO`、`GSPO`、`GMPO`、`SAPO`、`LUSPO`。

第三类，改的是梯度入口，也就是哪些 prompt、哪些 rollout、哪些 token 值得被学习。这一类包括 `SEED-GRPO`、`EDGE-GRPO`、`GFPO`。

第四类，改的是 reward 的定义对象，也就是 reward 究竟依赖单个回答还是整组回答的分布性质。这一类目前最典型的是 `GAPO`。

只有先接受这四层的区别，后面的实验设计才不会把不同层次的改动混在一起比较。

### 6.2 从 TRL 角度看哪些方法更适合直接起步

如果问题是“我现在就要搭实验，先从哪几条线开始”，一个比较务实的顺序是：

1. `DAPO` 作为长 CoT RL 的起始 baseline；
2. `Dr.GRPO` 作为长度偏置对照；
3. `CISPO`、`SAPO`、`LUSPO` 作为已经有明确 TRL 入口的结构性变体；
4. `GSPO` 作为 sequence-level 路线的组合实现；
5. `GFPO` 作为样本过滤路线的实验入口。

至于 `BNPO`、`GMPO`、`SEED-GRPO`、`EDGE-GRPO`、`GAPO`，它们更适合作为有明确问题意识的研究扩展来做，而不是不加区分地替换默认 baseline。

### 6.3 从公开评审信息看，哪些结论更稳

目前能核实到公开评审线索的主要有 `DAPO`、`Dr.GRPO`、`BNPO`、`SEED-GRPO` 四篇。把这些信息放在一起看，可以得到几个比较清楚的结论：

1. `DAPO` 被接受，说明社区确实认可系统 recipe 型工作的价值，尤其是在 reasoning RL 这种训练细节高度影响结果的场景里；
2. `Dr.GRPO` 被接受，说明长度偏置已经不是零散经验现象，而是一个得到明确关注的问题设定；
3. `BNPO` 的争议集中在理论语义，因此它更像一个值得继续追踪的研究方向，而不是已经形成共识的默认方案；
4. `SEED-GRPO` 的争议集中在 claim 与实现边界，说明不确定性建模这条线有价值，但命名和理论叙述需要更谨慎。

这些评审信息的共同指向是：相比口号式命名，社区更在意方法到底改了哪一层、理论语义是否自洽、以及实验结论是否与 claim 匹配。

### 6.4 更合理的实验组织方式

如果从研究问题出发，而不是从论文列表出发，我会优先做下面几组实验：

1. 长度偏置主线：`DAPO`、`Dr.GRPO`、`GSPO`、`LUSPO`；
2. surrogate 形状主线：`CISPO`、`SAPO`、`GMPO`；
3. 梯度入口主线：`GFPO`、`SEED-GRPO`、`EDGE-GRPO`；
4. reward-level 正交实验：在一条已经稳定的 optimizer 路线上叠加 `GAPO` 的 group-aware reward。

这样组织实验有一个直接好处：每组只回答一个问题，最后即便有收益差异，也比较容易追踪收益究竟来自哪一层改动。

### 6.5 一点点结论

GRPO 之后这些工作，并没有形成一条简单的单线演化谱系。更准确的说法是，它们共享同一个后训练背景，但分别从 advantage、归一化、ratio、样本选择和 reward 建模这些不同层次拆解问题。

因此阅读这类论文时，最重要的一步不是先看结果表，而是先回答一个更基本的问题：这个方法到底改了哪一层。这个问题如果答不清，后面的实现、调参和实验归因都会变得含混。

## 7. 参考资料

### 论文与官方资料

- TRL 仓库与文档：https://github.com/huggingface/trl
- TRL GRPO Trainer 文档：https://huggingface.co/docs/trl/grpo_trainer
- TRL PPO Trainer 文档：https://huggingface.co/docs/trl/ppo_trainer
- DeepSeekMath（GRPO 起源）：https://huggingface.co/papers/2402.03300
- DAPO：https://huggingface.co/papers/2503.14476
- Dr.GRPO / Understanding R1-Zero-Like Training：https://huggingface.co/papers/2503.20783
- BNPO：https://arxiv.org/abs/2506.02864
- MiniMax-M1 / CISPO：https://huggingface.co/papers/2506.13585
- GSPO：https://huggingface.co/papers/2507.18071
- GMPO：https://arxiv.org/abs/2507.20673
- GFPO：https://huggingface.co/papers/2508.09726
- SAPO：https://huggingface.co/papers/2511.20347
- LUSPO：https://arxiv.org/abs/2602.05261
- GAPO：https://arxiv.org/abs/2511.12596
- SEED-GRPO OpenReview：https://openreview.net/forum?id=6J6yU6n5Bp
- EDGE-GRPO：https://arxiv.org/abs/2507.21848